# Day13 — Multi-Model Router

This no-LLM acceptance notebook uses fake providers. It demonstrates deterministic complexity routing, format correction, and one cross-provider fallback without reading API keys.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'day13' else Path.cwd()))
from llm.model_router import ModelRouter, ProviderCallError, assess_complexity

class FakeProvider:
    def __init__(self, responses):
        self.responses = list(responses)
        self.calls = []
    def invoke(self, model, prompt):
        self.calls.append((model, prompt))
        response = self.responses.pop(0)
        if isinstance(response, Exception):
            raise response
        return response

In [ ]:
simple = assess_complexity('coder', {'files': [{'name': 'A.cs'}]})
complex_route = assess_complexity('coder', {'files': [{'name': f'F{i}.cs'} for i in range(4)]})
assert simple.level == 'simple'
assert complex_route.level == 'complex'
(simple, complex_route)

In [ ]:
primary = FakeProvider(['not-json', 'still-not-json'])
fallback = FakeProvider(['{"ok": true}'])
router = ModelRouter({'deepseek': primary, 'glm': fallback})
result = router.invoke(
    'architecture',
    'Return JSON',
    {},
    validator=lambda content: (content.startswith('{'), 'JSON required'),
)
assert result.record['format_retry_used'] is True
assert result.record['fallback_used'] is True
assert len(primary.calls) == 2 and len(fallback.calls) == 1
result.record

In [ ]:
primary = FakeProvider([
    ProviderCallError('timeout', 'MODEL_TIMEOUT', retryable=True),
    ProviderCallError('timeout', 'MODEL_TIMEOUT', retryable=True),
])
fallback = FakeProvider(['ok'])
router = ModelRouter({'deepseek': primary, 'glm': fallback})
result = router.invoke('architecture', 'Design', {})
assert result.content == 'ok'
assert [(item['provider'], item['requests']) for item in result.record['attempt_trace']] == [('deepseek', 2), ('glm', 1)]
result.record